# AI-Powered House Price Predictor: Exploratory Data Analysis & Model Benchmarking

This notebook documents the end-to-end Machine Learning research and benchmarking pipeline for predicting property values across major urban Pakistani real estate markets (Lahore, Islamabad, Karachi, Rawalpindi, Peshawar, Faisalabad, Multan, and Gujranwala).

### Pipeline Stages:
1. **Dataset Loading & Cleaning**
2. **Exploratory Data Analysis (EDA) & Market Visualizations**
3. **Preprocessing Pipeline Construction (ColumnTransformer)**
4. **Multi-Model Training (Linear Regression vs. Random Forest)**
5. **Model Evaluation ($R^2$, MAE, RMSE)**
6. **Pipeline Serialization with Joblib**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Dataset & Summary Statistics

In [ ]:
csv_path = os.path.join("..", "data", "housing_data.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join("data", "housing_data.csv")

df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape}")
df.head(10)

In [ ]:
df.info()
df.describe()

## 2. Exploratory Visualizations

In [ ]:
# Price Distribution in Crores (1 Crore = 10,000,000 PKR)
plt.figure(figsize=(10, 5))
sns.histplot(df['price'] / 1e7, kde=True, color='#10b981', bins=35)
plt.title("Distribution of Property Valuations (Crores PKR)", fontsize=14, fontweight='bold')
plt.xlabel("Price (Crores PKR)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Average Price by Location
plt.figure(figsize=(10, 5))
avg_loc = df.groupby('location')['price'].mean().sort_values(ascending=False) / 1e7
sns.barplot(x=avg_loc.index, y=avg_loc.values, palette='mako')
plt.title("Average House Valuation by City (Crores PKR)", fontsize=14, fontweight='bold')
plt.ylabel("Price (Crores PKR)")
plt.xlabel("City")
plt.xticks(rotation=30)
plt.show()

## 3. Preprocessing & ML Pipeline

In [ ]:
features = ['sqft', 'bedrooms', 'bathrooms', 'location']
target = 'price'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

numerical_cols = ['sqft', 'bedrooms', 'bathrooms']
categorical_cols = ['location']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

## 4. Model Training & Benchmark

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=16, min_samples_split=4, random_state=42, n_jobs=-1)
}

results = {}
pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    
    results[name] = {"R2": r2, "MAE": mae, "RMSE": rmse}
    pipelines[name] = pipe
    
    print(f"=== {name} ===")
    print(f"R2 Score : {r2:.4f}")
    print(f"MAE      : PKR {mae:,.2f}")
    print(f"RMSE     : PKR {rmse:,.2f}\n")

## 5. Model Selection & Export

In [ ]:
best_name = max(results, key=lambda k: results[k]["R2"])
print(f"Selected Production Model: {best_name}")

# Export Pipeline
out_model_path = os.path.join("..", "models", "house_price_model.joblib")
if not os.path.exists(os.path.dirname(out_model_path)):
    out_model_path = os.path.join("models", "house_price_model.joblib")

joblib.dump(pipelines[best_name], out_model_path)
print(f"Exported model pipeline to: {out_model_path}")